<a href="https://colab.research.google.com/github/NidhiCN24/Fake-News-Detection/blob/main/Fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Fake news classification model
#Installation
!pip -q install transformers datasets scikit-learn gradio

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import gradio as gr

In [ ]:
import os
import csv
import sys

In [ ]:
csv.field_size_limit(sys.maxsize)

df = pd.read_csv(
    '/content/WELFake_Dataset.csv',
    engine='python',
    encoding='utf-8',
    sep=',',
    quotechar='"',
    escapechar='\\',
    on_bad_lines='warn'
)

# Basic cleaning
df = df[['text','label']].dropna()
#Removing extremely short rows as they do not contribute much to the model understanding and leads to increased training time
df = df[df['text'].str.len() > 20]
df = df.sample(15000, random_state=42)
#converting the label to integers (as earlier I got float value as the label)
df['label'] = df['label'].astype(int)

print(df.head())
print("\nLabel Distribution:\n", df['label'].value_counts())

In [ ]:
#train test split
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df).map(tokenize, batched=True)

train_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])
val_ds.set_format(type='torch', columns=['input_ids','attention_mask','label'])

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

model.config.id2label = {0: "Real News", 1: "Fake News"}
model.config.label2id = {"Real News": 0, "Fake News": 1}

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": (preds == labels).mean()
    }

In [ ]:
#training the model
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    logging_steps=100,
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
#model evaluation
preds = trainer.predict(val_ds)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print(classification_report(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

In [ ]:
#samples that are misclassified
val_df = val_df.reset_index(drop=True)
wrong = np.where(y_pred != y_true)[0][:5]

print("\nMisclassified Samples:\n")
for i in wrong:
    print("Text:", val_df['text'][i][:200])
    print("Actual:", y_true[i], "Pred:", y_pred[i])
    print()

In [ ]:
#saving the model
model.save_pretrained('/content/fake_news_model')
tokenizer.save_pretrained('/content/fake_news_model')

In [ ]:
#improvement - weighted loss
class_counts = df['label'].value_counts()
weights = torch.tensor([1/class_counts[0], 1/class_counts[1]], dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fn = torch.nn.CrossEntropyLoss(weight=weights.to(model.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer_w = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)
trainer_w.train()

In [ ]:
print(model.config.id2label)

In [ ]:
import torch

def predict(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    # Move to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=1).item()
    if(pred == 1):
      final_prediction = "Real News"
    else:
      final_prediction = "Fake News"

    return final_prediction

import gradio as gr

iface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Enter news text here..."),
    outputs="text"
)

iface.launch(share=True)